In [ ]:
from mlflow.pytorch import load_state_dict
checkpoint_path = "./mlruns/Quora-Question-Pairs/LSTM_attention/e079c52d606c470cac78fd2e693800db/artifacts/"
state_dict = load_state_dict(checkpoint_path)  # path to the artifact URI
model.load_state_dict(state_dict)

In [ ]:
# ---- DIAGNOSTIC PATCH ----
def forward_diagnostic(self, q1, q2):
    """
    Temporary forward pass that returns logits AND cosine similarities.
    We'll replace the model's forward with this for analysis.
    """
    # Replicate the exact forward logic from your current model
    h1_raw = self._encode(q1)
    h2_raw = self._encode(q2)
    
    h1 = self.proj(h1_raw)
    h2 = self.proj(h2_raw)
    
    # Normalization (as in your current setup)
    h1_norm = F.normalize(h1, p=2, dim=-1)
    h2_norm = F.normalize(h2, p=2, dim=-1)
    
    diff = torch.abs(h1_norm - h2_norm)
    prod = h1_norm * h2_norm
    cosine_sim = (h1_norm * h2_norm).sum(dim=-1, keepdim=True)
    
    feat = torch.cat([diff, prod, cosine_sim], dim=1)
    logits = self.fc(feat).squeeze(1)
    
    return {
        "logits": logits,
        "cosine": cosine_sim.squeeze(1),   # shape (batch,)
    }

# Temporarily bind the diagnostic method
import types
model.forward_diagnostic = types.MethodType(forward_diagnostic, model)
print(">>> Diagnostic forward method attached.")

In [ ]:
@torch.no_grad()
def run_phase1_diagnostic(model, dataloader, device):
    model.eval()
    all_labels = []
    all_probs = []
    all_cosines = []
    
    for batch in tqdm(dataloader, desc="Phase 1 Diagnostic"):
        q1 = batch["q1"].to(device)
        q2 = batch["q2"].to(device)
        labels = batch["label"].cpu().numpy()
        
        outputs = model.forward_diagnostic(q1, q2)
        probs = torch.sigmoid(outputs["logits"]).cpu().numpy()
        cosines = outputs["cosine"].cpu().numpy()
        
        all_labels.extend(labels)
        all_probs.extend(probs)
        all_cosines.extend(cosines)
    
    import pandas as pd
    import numpy as np
    
    df = pd.DataFrame({
        "label": all_labels,
        "prob": all_probs,
        "cosine": all_cosines
    })
    
    # Analysis
    fp = df[(df["label"] == 0) & (df["prob"] > 0.5)]
    tp = df[(df["label"] == 1) & (df["prob"] > 0.5)]
    tn = df[(df["label"] == 0) & (df["prob"] <= 0.5)]
    fn = df[(df["label"] == 1) & (df["prob"] <= 0.5)]
    
    print("\n" + "="*60)
    print("PHASE 1 DIAGNOSTIC RESULTS")
    print("="*60)
    print(f"Total validation samples: {len(df)}")
    print(f"False Positives (FP): {len(fp)}  ({len(fp)/len(df)*100:.2f}%)")
    print(f"True Positives  (TP): {len(tp)}  ({len(tp)/len(df)*100:.2f}%)")
    print(f"True Negatives  (TN): {len(tn)}  ({len(tn)/len(df)*100:.2f}%)")
    print(f"False Negatives (FN): {len(fn)}  ({len(fn)/len(df)*100:.2f}%)")
    print("\n--- Cosine Similarity Statistics ---")
    print(f"All negatives (label=0): mean={df[df['label']==0]['cosine'].mean():.4f} ± {df[df['label']==0]['cosine'].std():.4f}")
    print(f"False Positives only  : mean={fp['cosine'].mean():.4f} ± {fp['cosine'].std():.4f}")
    print(f"True Positives only   : mean={tp['cosine'].mean():.4f} ± {tp['cosine'].std():.4f}")
    print(f"All positives (label=1): mean={df[df['label']==1]['cosine'].mean():.4f} ± {df[df['label']==1]['cosine'].std():.4f}")
    print("\n--- Cosine Distribution Overlap ---")
    # Binned counts for positives and negatives
    bins = np.linspace(-1, 1, 21)
    pos_cos = df[df['label']==1]['cosine']
    neg_cos = df[df['label']==0]['cosine']
    print("Bin       Pos_count   Neg_count")
    for i in range(len(bins)-1):
        low, high = bins[i], bins[i+1]
        p_cnt = ((pos_cos >= low) & (pos_cos < high)).sum()
        n_cnt = ((neg_cos >= low) & (neg_cos < high)).sum()
        if p_cnt > 0 or n_cnt > 0:
            print(f"[{low:.2f}, {high:.2f}): {p_cnt:6d}      {n_cnt:6d}")
    
    return df

# Run it
df_diag = run_phase1_diagnostic(model, val_dataloader, sys_cfg.DEVICE)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

@torch.no_grad()
def analyze_probability_distribution(model, dataloader, device):
    model.eval()
    all_probs = []
    all_labels = []
    
    for batch in tqdm(dataloader, desc="Collecting probabilities"):
        q1 = batch["q1"].to(device)
        q2 = batch["q2"].to(device)
        labels = batch["label"].cpu().numpy()
        
        logits = model(q1, q2)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        all_probs.extend(probs)
        all_labels.extend(labels)
    
    probs = np.array(all_probs)
    labels = np.array(all_labels)
    
    # Histograms
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.hist(probs[labels==1], bins=30, alpha=0.5, label='Positives (Duplicates)', density=True)
    plt.hist(probs[labels==0], bins=30, alpha=0.5, label='Negatives (Non-duplicates)', density=True)
    plt.xlabel('Predicted Probability')
    plt.ylabel('Density')
    plt.legend()
    plt.title('Probability Distributions by Class')
    
    # Calibration curve
    prob_true, prob_pred = calibration_curve(labels, probs, n_bins=15)
    plt.subplot(1,2,2)
    plt.plot(prob_pred, prob_true, marker='o', label='Model')
    plt.plot([0,1],[0,1], '--', label='Perfect Calibration')
    plt.xlabel('Mean Predicted Probability')
    plt.ylabel('Fraction of Positives')
    plt.legend()
    plt.title('Calibration Curve (Reliability Diagram)')
    plt.tight_layout()
    plt.show()
    
    # Statistics for high-confidence false positives
    high_conf_fp = probs[(labels==0) & (probs > 0.9)]
    print(f"\nHigh-confidence (>0.9) false positives: {len(high_conf_fp)} out of {len(probs[labels==0])} negatives ({len(high_conf_fp)/len(probs[labels==0])*100:.2f}%)")
    
    return probs, labels

probs, labels = analyze_probability_distribution(model, val_dataloader, sys_cfg.DEVICE)

In [ ]:
@torch.no_grad()
def encode_with_attention(model, question):
    """
    Manually run the encoder steps and return both context vector and attention weights.
    """
    # Embedding + norm + dropout
    emb = model.embedding(question)
    emb = model.emb_norm(emb)
    emb = model.emb_dropout(emb)
    
    # LSTM
    out, _ = model.LSTM(emb)  # (batch, seq, hidden*dir)
    
    # Attention (manually compute)
    mask = (question != 0).float()  # (batch, seq)
    energy = torch.tanh(model.attention.W(out))
    scores = model.attention.V(energy).squeeze(-1)  # (batch, seq)
    scores = scores.masked_fill(mask == 0, model.attention.mask_fill_num)
    weights = F.softmax(scores, dim=-1)  # (batch, seq)
    weights = model.attention.dropout(weights)
    lstm_masked = out * mask.unsqueeze(-1)
    context = torch.bmm(weights.unsqueeze(1), lstm_masked).squeeze(1)
    
    return context, weights

In [ ]:
def analyze_attention_manual(model, dataloader, device, idx2word, num_batches=10):
    model.eval()
    results = []
    
    for batch_idx, batch in enumerate(dataloader):
        if batch_idx >= num_batches:
            break
        q1 = batch["q1"].to(device)
        labels = batch["label"].cpu().numpy()
        
        # Get context and attention weights
        _, attn_weights = encode_with_attention(model, q1)  # (batch, seq_len)
        
        for i in range(q1.size(0)):
            tokens = [idx2word[idx.item()] for idx in q1[i].cpu() if idx.item() != 0]
            if not tokens:
                continue
            weights = attn_weights[i, :len(tokens)].cpu().numpy()
            top3_idx = np.argsort(weights)[::-1][:3]
            results.append({
                'label': labels[i],
                'tokens': tokens,
                'top_tokens': [tokens[j] for j in top3_idx],
                'top_weights': [weights[j] for j in top3_idx]
            })
    
    from collections import Counter
    all_top_tokens = [tok for r in results for tok in r['top_tokens']]
    token_counts = Counter(all_top_tokens)
    
    print("=== Top Attended Tokens (across all samples) ===")
    for tok, count in token_counts.most_common(20):
        print(f"  {tok}: {count}")
    
    pos_tokens = [tok for r in results if r['label']==1 for tok in r['top_tokens']]
    neg_tokens = [tok for r in results if r['label']==0 for tok in r['top_tokens']]
    pos_counter = Counter(pos_tokens)
    neg_counter = Counter(neg_tokens)
    
    print("\nTop tokens in POSITIVES:")
    for tok, count in pos_counter.most_common(10):
        print(f"  {tok}: {count}")
    print("\nTop tokens in NEGATIVES:")
    for tok, count in neg_counter.most_common(10):
        print(f"  {tok}: {count}")
    
    return results

# Run it
attn_results = analyze_attention_manual(model, val_dataloader, sys_cfg.DEVICE, idx2word)

In [ ]:
@torch.no_grad()
def forward_with_features(model, q1, q2):
    """
    Runs the model forward pass and returns:
      - logits
      - fc_input (the concatenated [diff, prod, cosine] vector)
    """
    h1 = model.proj(model._encode(q1))
    h2 = model.proj(model._encode(q2))
    
    h1_norm = F.normalize(h1, p=2, dim=-1)
    h2_norm = F.normalize(h2, p=2, dim=-1)
    
    diff = torch.abs(h1_norm - h2_norm)
    prod = h1_norm * h2_norm
    cosine = (h1_norm * h2_norm).sum(dim=-1, keepdim=True)
    
    fc_input = torch.cat([diff, prod, cosine], dim=1)
    logits = model.fc(fc_input).squeeze(1)
    
    return logits, fc_input

In [ ]:
def remove_all_hooks(module):
    """Recursively remove all hooks from a PyTorch module."""
    # Clear forward hooks
    module._forward_hooks.clear()
    module._forward_pre_hooks.clear()
    # Clear backward hooks
    module._backward_hooks.clear()
    module._backward_pre_hooks.clear()
    # Recursively apply to children
    for child in module.children():
        remove_all_hooks(child)

# Apply to the entire model
remove_all_hooks(model)
print("All hooks purged from model.")

In [ ]:
# Test with a single batch
test_batch = next(iter(val_dataloader))
q1 = test_batch["q1"].to(sys_cfg.DEVICE)
q2 = test_batch["q2"].to(sys_cfg.DEVICE)
logits = model(q1, q2)
print(f"Test forward pass successful. Logits shape: {logits.shape}")

In [ ]:
import numpy as np
from tqdm import tqdm

@torch.no_grad()
def extract_all_features(model, dataloader, device):
    model.eval()
    all_logits = []
    all_fc_inputs = []
    all_labels = []
    
    for batch in tqdm(dataloader, desc="Extracting features"):
        q1 = batch["q1"].to(device)
        q2 = batch["q2"].to(device)
        labels = batch["label"].cpu().numpy()
        
        logits, fc_input = forward_with_features(model, q1, q2)
        
        all_logits.append(logits.cpu().numpy())
        all_fc_inputs.append(fc_input.cpu().numpy())
        all_labels.append(labels)
    
    logits = np.concatenate(all_logits)
    fc_inputs = np.concatenate(all_fc_inputs)
    labels = np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    
    return fc_inputs, labels, probs

# Run extraction
fc_inputs, labels, probs = extract_all_features(model, val_dataloader, sys_cfg.DEVICE)
print(f"Extracted features shape: {fc_inputs.shape}")  # Should be (N, 257) for PROJ_DIM=128

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

proj_dim = model_cfg.PROJECT_DIM  # 128

# Split the fc_input into components
diff_feat = fc_inputs[:, :proj_dim]
prod_feat = fc_inputs[:, proj_dim:2*proj_dim]
cos_feat = fc_inputs[:, -1]  # last column

# Compute L2 norms of diff and prod as summary statistics
df_stats = pd.DataFrame({
    'label': labels,
    'cos': cos_feat,
    'diff_norm': np.linalg.norm(diff_feat, axis=1),
    'prod_norm': np.linalg.norm(prod_feat, axis=1)
})

print("=== Feature Statistics by Class ===")
for cls, name in [(1, 'Positives'), (0, 'Negatives')]:
    subset = df_stats[df_stats['label'] == cls]
    print(f"\n{name}:")
    print(f"  cos mean ± std:        {subset['cos'].mean():.4f} ± {subset['cos'].std():.4f}")
    print(f"  diff_norm mean ± std:  {subset['diff_norm'].mean():.4f} ± {subset['diff_norm'].std():.4f}")
    print(f"  prod_norm mean ± std:  {subset['prod_norm'].mean():.4f} ± {subset['prod_norm'].std():.4f}")

# Check how well these features alone can classify (before FC layers)
print("\n=== Cross-validated F1 using ONLY raw comparator features ===")
clf = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
scores = cross_val_score(clf, fc_inputs, labels, cv=5, scoring='f1')
print(f"5-fold CV F1: {scores.mean():.4f} ± {scores.std():.4f}")

# Compare with using only cosine
scores_cos_only = cross_val_score(clf, cos_feat.reshape(-1,1), labels, cv=5, scoring='f1')
print(f"Using only cosine: {scores_cos_only.mean():.4f} ± {scores_cos_only.std():.4f}")

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce to 2D
pca = PCA(n_components=2)
fc_2d = pca.fit_transform(fc_inputs)

# Plot all points
plt.figure(figsize=(10,6))
for label, color, name in [(0, 'red', 'Negatives'), (1, 'blue', 'Positives')]:
    mask = labels == label
    plt.scatter(fc_2d[mask,0], fc_2d[mask,1], c=color, label=name, alpha=0.3, s=2)
plt.legend()
plt.title('PCA of Comparator Features (Input to FC)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.show()

# Highlight False Positives (label=0, prob>0.5)
fp_mask = (labels == 0) & (probs > 0.5)
tp_mask = (labels == 1) & (probs > 0.5)
tn_mask = (labels == 0) & (probs <= 0.5)
fn_mask = (labels == 1) & (probs <= 0.5)

plt.figure(figsize=(10,6))
plt.scatter(fc_2d[tn_mask,0], fc_2d[tn_mask,1], c='gray', alpha=0.2, s=2, label='True Negatives')
plt.scatter(fc_2d[tp_mask,0], fc_2d[tp_mask,1], c='blue', alpha=0.3, s=2, label='True Positives')
plt.scatter(fc_2d[fp_mask,0], fc_2d[fp_mask,1], c='red', alpha=0.8, s=8, label='False Positives')
plt.scatter(fc_2d[fn_mask,0], fc_2d[fn_mask,1], c='orange', alpha=0.8, s=8, label='False Negatives')
plt.legend()
plt.title('PCA with Error Types Highlighted')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
plt.show()

In [ ]:
first_fc = model.fc[0]  # nn.Linear
weights = first_fc.weight.data.cpu().numpy()  # shape: (FC_DIMS[0], input_dim)
input_dim = weights.shape[1]

# Average absolute weight per input feature
avg_abs_weight = np.abs(weights).mean(axis=0)

# Split by component
diff_w = avg_abs_weight[:proj_dim]
prod_w = avg_abs_weight[proj_dim:2*proj_dim]
cos_w = avg_abs_weight[-1]

print("=== Average Absolute Weight by Component ===")
print(f"diff features:  mean={diff_w.mean():.4f}, std={diff_w.std():.4f}")
print(f"prod features:  mean={prod_w.mean():.4f}, std={prod_w.std():.4f}")
print(f"cosine feature: {cos_w:.4f}")

In [ ]:
import pandas as pd
from pathlib import Path

train_df = pd.read_csv(PathConfig.TRAIN_CSV_PATH)

# Create canonical pair representation (order-independent)
train_df['q_pair'] = train_df.apply(
    lambda r: tuple(sorted([str(r['question1']).strip(), str(r['question2']).strip()])), 
    axis=1
)

# Find pairs that appear with different labels
conflict_counts = train_df.groupby('q_pair')['is_duplicate'].nunique()
conflicting = conflict_counts[conflict_counts > 1]

print(f"Total training samples: {len(train_df)}")
print(f"Unique question pairs: {train_df['q_pair'].nunique()}")
print(f"Conflicting pairs (same text, different labels): {len(conflicting)}")

if len(conflicting) > 0:
    print("\nExample conflicting pairs:")
    for pair in conflicting.index[:5]:
        subset = train_df[train_df['q_pair'] == pair]
        print(f"\nPair: '{pair[0][:60]}...' <-> '{pair[1][:60]}...'")
        for _, row in subset.iterrows():
            print(f"  Label: {row['is_duplicate']}")

In [ ]:
pos_weight = train_dataset.pos_class_weight(sys_cfg.DEVICE)
print(f"\nComputed pos_weight: {pos_weight.item():.4f}")

# Effective probability threshold shift
import torch
eff_logit_shift = -np.log(pos_weight.item())
eff_prob_thresh = torch.sigmoid(torch.tensor(eff_logit_shift)).item()
print(f"Loss-optimal probability threshold (with BCEWithLogitsLoss + pos_weight): {eff_prob_thresh:.3f}")
print(f"This means the model is encouraged to predict POSITIVE whenever P > {eff_prob_thresh:.3f}")
print(f"(A standard 0.5 threshold corresponds to pos_weight = 1.0)")

In [ ]:
@torch.no_grad()
def analyze_by_length(model, dataloader, device):
    model.eval()
    results = []
    for batch in dataloader:
        q1 = batch["q1"].to(device)
        q2 = batch["q2"].to(device)
        labels = batch["label"].cpu().numpy()
        logits = model(q1, q2)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        len1 = (q1 != 0).sum(dim=1).cpu().numpy()
        len2 = (q2 != 0).sum(dim=1).cpu().numpy()
        avg_len = (len1 + len2) / 2
        
        for i in range(len(labels)):
            results.append({
                'label': labels[i],
                'prob': probs[i],
                'avg_len': avg_len[i],
                'pred': probs[i] > 0.5
            })
    df = pd.DataFrame(results)
    
    bins = [0, 8, 12, 16, 20, 30, 50]
    df['len_bin'] = pd.cut(df['avg_len'], bins)
    
    print("Performance by average question length (in tokens):")
    print("Length Bin   | Count | Precision | Recall | F1")
    print("-" * 50)
    for bin_name, group in df.groupby('len_bin', observed=True):
        if len(group) < 50:
            continue
        tp = ((group['pred'] == 1) & (group['label'] == 1)).sum()
        fp = ((group['pred'] == 1) & (group['label'] == 0)).sum()
        fn = ((group['pred'] == 0) & (group['label'] == 1)).sum()
        precision = tp / (tp + fp) if (tp+fp) > 0 else 0
        recall = tp / (tp + fn) if (tp+fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision+recall) > 0 else 0
        print(f"{str(bin_name):>12} | {len(group):5} | {precision:.4f}   | {recall:.4f} | {f1:.4f}")
    
    return df

len_df = analyze_by_length(model, val_dataloader, sys_cfg.DEVICE)

In [ ]:
@torch.no_grad()
def get_all_distances(model, loader, device):
    model.eval()
    all_dists, all_labels = [], []
    for batch in tqdm(loader, desc="Computing distances"):
        q1 = batch["q1"].to(device)
        q2 = batch["q2"].to(device)
        labels = batch["label"].to(device).long()

        dist = model(q1, q2)               # Euclidean distance
        all_dists.append(dist.cpu())
        all_labels.append(labels.cpu())

    return torch.cat(all_dists), torch.cat(all_labels)

all_dists, all_labels = get_all_distances(model, val_dataloader, sys_cfg.DEVICE)

In [ ]:
pos_dists = all_dists[all_labels == 1]
neg_dists = all_dists[all_labels == 0]

print(f"Number of positive pairs: {len(pos_dists)}")
print(f"Number of negative pairs: {len(neg_dists)}")

print("\n--- Distance distribution ---")
print(f"Pos  mean: {pos_dists.mean():.4f}, std: {pos_dists.std():.4f}")
print(f"Neg  mean: {neg_dists.mean():.4f}, std: {neg_dists.std():.4f}")

In [ ]:
margin = model_cfg.MARGIN   # should be 1.0
neg_inside = (neg_dists < margin).sum().item()
print(f"Negative pairs with distance < margin: {neg_inside} ({neg_inside/len(neg_dists)*100:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(pos_dists, bins=50, alpha=0.5, label='Positives', density=True)
plt.hist(neg_dists, bins=50, alpha=0.5, label='Negatives', density=True)
plt.axvline(margin, color='r', linestyle='--', label=f'Margin = {margin}')
plt.xlabel('Euclidean Distance')
plt.ylabel('Density')
plt.legend()
plt.title('Distance distribution of validation pairs')
plt.show()

In [ ]:
# For negatives (label=0), sort by distance ascending
neg_sort_indices = torch.argsort(neg_dists)
worst_neg_indices = torch.where(all_labels == 0)[0][neg_sort_indices[:20]]  # top 20 false positives

# Make sure val_df is the DataFrame that was used to create the Validation Dataset
# (it must contain the columns 'question1' and 'question2')

for global_idx in worst_neg_indices.tolist():
    row = val_df.iloc[global_idx]          # global_idx matches DataLoader order
    dist_val = all_dists[global_idx].item()
    print(f"Dist: {dist_val:.4f}")
    print(f"Q1: {row['question1']}")
    print(f"Q2: {row['question2']}")
    print("---")

In [ ]:
with torch.no_grad():
    batch = next(iter(val_dataloader))
    q1 = batch['q1'].to(sys_cfg.DEVICE)
    h = model._encode(q1[:32])   # take a batch
    print("Variance per dimension:", h.var(dim=0).mean().item())

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

model.eval()
device = next(model.parameters()).device

threshold = 0.5457           # your optimal threshold
num_examples_to_show = 10    # how many false positives to print

batch = next(iter(val_dataloader))
q1 = batch['q1'].to(device)
q2 = batch['q2'].to(device)
labels = batch['label'].to(device)

with torch.no_grad():
    distances = model(q1, q2)
    probs = torch.exp(-distances)
    preds = (probs >= threshold).long()

    # False positives: predicted duplicate but label 0
    fp_mask = (preds == 1) & (labels == 0)
    fp_indices = torch.where(fp_mask)[0].cpu().tolist()

    if len(fp_indices) == 0:
        print("No false positives in this batch. Try another batch or lower threshold.")
    else:
        # Sort by distance ascending (worst first)
        fp_dists = distances[fp_indices].cpu()
        sorted_fp = sorted(zip(fp_indices, fp_dists), key=lambda x: x[1])

        for idx, (global_idx, dist_val) in enumerate(sorted_fp[:num_examples_to_show]):
            print(f"\n{'='*100}")
            print(f"Example {idx+1} | Distance: {dist_val:.6f} | Prob: {probs[global_idx]:.4f} | True label: 0")
            print()

            # Extract tokens
            q1_tokens = [idx2word.get(tok.item(), '<unk>') for tok in q1[global_idx]]
            q2_tokens = [idx2word.get(tok.item(), '<unk>') for tok in q2[global_idx]]

            # Get attention weights (use the helper from previous answer)
            ctx1, attn1 = get_attention_and_context(model, q1[global_idx:global_idx+1])
            ctx2, attn2 = get_attention_and_context(model, q2[global_idx:global_idx+1])

            # Print Q1
            print("Q1 tokens and attention:")
            for token, weight in zip(q1_tokens, attn1):
                if token != '[PAD]':   # skip padding tokens for brevity
                    print(f"  {token:25s} : {weight:.4f}")
            # Print Q2
            print("\nQ2 tokens and attention:")
            for token, weight in zip(q2_tokens, attn2):
                if token != '[PAD]':
                    print(f"  {token:25s} : {weight:.4f}")

    # ------------------------------------------------------------------
    # Also show a few true positives for contrast
    tp_mask = (preds == 1) & (labels == 1)
    tp_indices = torch.where(tp_mask)[0].cpu().tolist()
    if len(tp_indices) > 0:
        print("\n\n--- For contrast: a true positive (label=1, predicted 1) ---")
        sample_tp = tp_indices[0]   # first true positive
        q1_tokens_tp = [idx2word.get(tok.item(), '<unk>') for tok in q1[sample_tp]]
        q2_tokens_tp = [idx2word.get(tok.item(), '<unk>') for tok in q2[sample_tp]]
        ctx1_tp, attn1_tp = get_attention_and_context(model, q1[sample_tp:sample_tp+1])
        ctx2_tp, attn2_tp = get_attention_and_context(model, q2[sample_tp:sample_tp+1])
        print("Q1 tokens and attention:")
        for token, weight in zip(q1_tokens_tp, attn1_tp):
            if token != '[PAD]':
                print(f"  {token:25s} : {weight:.4f}")
        print("Q2 tokens and attention:")
        for token, weight in zip(q2_tokens_tp, attn2_tp):
            if token != '[PAD]':
                print(f"  {token:25s} : {weight:.4f}")

    # ------------------------------------------------------------------
    # Variance checks on the whole batch (first 64 samples to save time)
    with torch.no_grad():
        # LSTM output before attention
        emb = model.embedding(q1[:64])
        if model.config.LAYER_NORM_EMB: emb = model.emb_norm(emb)
        emb = model.emb_dropout(emb)
        lstm_out, _ = model.LSTM(emb)
        var_lstm = lstm_out.var(dim=[0,1]).mean().item()
        print(f"\nLSTM output variance (pre‑attention): {var_lstm:.6f}")

        # Final context vector variance
        h = model._encode(q1[:64])
        var_ctx = h.var(dim=0).mean().item()
        print(f"Context vector variance: {var_ctx:.6f}")